In [1]:
import networkx as nx
import numpy as np
from gwgraphs import *

In [2]:
def read_graph_data():
    # Read the adjacency matrix (edges)
    with open("BZR_A.txt", "r") as f:
        edges = [tuple(map(int, line.split(','))) for line in f]

    # Read graph indicators
    with open("BZR_graph_indicator.txt", "r") as f:
        graph_indicators = [int(line.strip()) for line in f if line.strip()]

    # Read graph labels
    with open("BZR_graph_labels.txt", "r") as f:
        graph_labels = [int(line.strip()) for line in f]

    # Read node labels
    with open("BZR_node_labels.txt", "r") as f:
        node_labels = [int(line.strip()) for line in f]

    # Read node attributes
    with open("BZR_node_attributes.txt", "r") as f:
        node_attributes = [
            list(map(float, line.strip().split(','))) for line in f if line.strip()
        ]

    return edges, graph_indicators, graph_labels, node_labels, node_attributes


def construct_networkx_graph(edges, graph_indicators, graph_labels,
                             node_labels, node_attributes):
    graphs = {}
    graphs_normalized = {}
    
    # Normalize all node attributes
    normalized_attributes = normalize_features(node_attributes)
    
    for node_id, graph_id in enumerate(graph_indicators, start=1):
        if graph_id not in graphs:
            graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])
            graphs_normalized[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

    for edge in edges:
        node1, node2 = edge
        graph_id = graph_indicators[
            node1 - 1]  # Determine which graph the edge belongs to
        graphs[graph_id].add_edge(node1, node2)
        graphs_normalized[graph_id].add_edge(node1, node2)

    for node_id, (graph_id, node_label,
                  node_attr, norm_attr) in enumerate(zip(graph_indicators, node_labels,
                                              node_attributes, normalized_attributes),
                                          start=1):
        # Original graphs with original features
        graphs[graph_id].nodes[node_id]['label'] = node_label
        graphs[graph_id].nodes[node_id]['attributes'] = node_attr
        
        # Normalized graphs with normalized features
        graphs_normalized[graph_id].nodes[node_id]['label'] = node_label
        graphs_normalized[graph_id].nodes[node_id]['attributes'] = norm_attr

    return graphs, graphs_normalized


def compute_bzr_distance_matrix(graph):
    num_nodes = len(graph.nodes)
    distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

    # Get the correct node indices to be used for indexing the matrix
    node_list = list(graph.nodes())
    node_index = {node: i for i, node in enumerate(node_list)}

    for node in graph.nodes:
        lengths = nx.shortest_path_length(graph, source=node, weight="weight")

        for target, length in lengths.items():
            if target in node_index:
                distance_matrix[node_index[node], node_index[target]] = length

    return distance_matrix


def normalize_features(features):
    features = np.array(features)
    # Normalize each feature dimension independently
    mean = np.mean(features, axis=0)
    std = np.std(features, axis=0)
    # Avoid division by zero
    std[std == 0] = 1
    normalized_features = (features - mean) / std
    return normalized_features.tolist()

In [3]:
edges, graph_indicators, graph_labels, node_labels, node_attributes = read_graph_data()
graphs, graphs_normalized = construct_networkx_graph(edges, graph_indicators, graph_labels,
                                                   node_labels, node_attributes)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
        )

Graph ID: 1, number of nodes: 30, label: -1
Graph ID: 2, number of nodes: 33, label: -1
Graph ID: 3, number of nodes: 30, label: -1
Graph ID: 4, number of nodes: 33, label: -1
Graph ID: 5, number of nodes: 30, label: -1
Graph ID: 6, number of nodes: 33, label: -1
Graph ID: 7, number of nodes: 30, label: -1
Graph ID: 8, number of nodes: 30, label: -1
Graph ID: 9, number of nodes: 33, label: -1
Graph ID: 10, number of nodes: 30, label: -1
Graph ID: 11, number of nodes: 30, label: -1
Graph ID: 12, number of nodes: 30, label: -1
Graph ID: 13, number of nodes: 33, label: -1
Graph ID: 14, number of nodes: 33, label: -1
Graph ID: 15, number of nodes: 33, label: -1
Graph ID: 16, number of nodes: 33, label: -1
Graph ID: 17, number of nodes: 32, label: -1
Graph ID: 18, number of nodes: 32, label: -1
Graph ID: 19, number of nodes: 32, label: -1
Graph ID: 20, number of nodes: 35, label: -1
Graph ID: 21, number of nodes: 32, label: -1
Graph ID: 22, number of nodes: 35, label: -1
Graph ID: 23, numbe

In [4]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_bzr_distance_matrix(graph) for graph in graphs_list]

In [5]:
"""fGW-NCC: alpha hyperparameter tuning"""

'fGW-NCC: alpha hyperparameter tuning'

In [7]:
### TEST 1: Use dataset average number of nodes ###
alpha_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
alpha_tests = {
        alph: []
        for alph in alpha_list
    }  # dictionary to store accuracies for each k
for alpha in alpha_list:
    print()
    print("Currently testing alpha =", alpha)
    print()
    bzr_exp = fgw_barycenter_alg_alpha(distance_matrices, 
                                       graphs_list, 
                                       labels_list, 
                                       num_folds=10, 
                                       num_nodes_negone=36, 
                                       num_nodes_one=36,
                                       alpha=alpha,
                                       method="stratk")
    #classification_accuracy = sum(bzr_exp[0])/len(bzr_exp[0]) * 100
    alpha_tests[alpha].append(bzr_exp)


Currently testing alpha = 0.0

Stratified k-fold CV selected
Fold 0:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.9532900255820174  Fold 0
F1 Score (weighted avg): 70.78476679503638%, Fold 0
Classification Accuracy: 68.29268292682927% Fold 0
Classification Accuracy for Class -1: 68.75% Fold 0
Classification Accuracy for Class 1: 66.66666666666666% Fold 0
Fold 1:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.9683058970165135  Fold 1
F1 Score (weighted avg): 66.55664344541948%, Fold 1
Classification Accuracy: 63.41463414634146% Fold 1
Classification Accuracy for Class -1: 59.375% Fold 1
Classification Accuracy for Class 1: 77.77777777777779% Fold 1
Fold 2:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  1.0446590486354097  Fold 2
F1 Score (weighted avg): 70.78476679503638%, Fol

In [18]:
best_alpha = alpha_list[0]
best_alpha_acc = 0
best_alpha_f1 = 0
alph_avg_accuracies = {}
alph_avg_f1_scores = {}
classes = [-1, 1]

for alph in alpha_list:
        avg_acc = sum(alpha_tests[alph][0][0]) / len(alpha_tests[alph][0][0])
        avg_f1 = sum(alpha_tests[alph][0][3]) / len(alpha_tests[alph][0][3])

        alph_avg_accuracies[alph] = {
            'overall':
            avg_acc,
            'class_1':
            sum(alpha_tests[alph][0][1]) / len(alpha_tests[alph][0][1]),
            'class_-1':
            sum(alpha_tests[alph][0][2]) / len(alpha_tests[alph][0][2])
        }
        alph_avg_f1_scores[alph] = avg_f1

        print(f"\nalpha={alph} Average Overall Accuracy: {avg_acc:.2f}%")
        print(f"alpha={alph} Average F1 Score (weighted avg): {avg_f1:.2f}%")
        for class_label in [1, -1]:
            print(
                f"alpha={alph} Average Class {class_label} Accuracy: {alph_avg_accuracies[alph][f'class_{class_label}']:.2f}%"
            )

        if avg_acc > best_alpha_acc:
            best_alpha_acc = avg_acc
            best_alpha = alph

        if avg_f1 > best_alpha_f1:
            best_alpha_f1 = avg_f1

print(f"\nBest performance achieved with alpha={best_alpha}:")
print(f"Overall Accuracy: {alph_avg_accuracies[best_alpha]['overall']:.2f}%")
print(f"F1 Score (weighted avg): {alph_avg_f1_scores[best_alpha]:.2f}%")
print(f"Class 1 Accuracy: {alph_avg_accuracies[best_alpha]['class_1']:.2f}%")
print(f"Class -1 Accuracy: {alph_avg_accuracies[best_alpha]['class_-1']:.2f}%")


alpha=0.0 Average Overall Accuracy: 65.66%
alpha=0.0 Average F1 Score (weighted avg): 68.65%
alpha=0.0 Average Class 1 Accuracy: 64.89%
alpha=0.0 Average Class -1 Accuracy: 68.33%

alpha=0.1 Average Overall Accuracy: 69.63%
alpha=0.1 Average F1 Score (weighted avg): 72.16%
alpha=0.1 Average Class 1 Accuracy: 69.27%
alpha=0.1 Average Class -1 Accuracy: 70.83%

alpha=0.2 Average Overall Accuracy: 69.38%
alpha=0.2 Average F1 Score (weighted avg): 71.89%
alpha=0.2 Average Class 1 Accuracy: 69.59%
alpha=0.2 Average Class -1 Accuracy: 68.61%

alpha=0.3 Average Overall Accuracy: 70.62%
alpha=0.3 Average F1 Score (weighted avg): 73.02%
alpha=0.3 Average Class 1 Accuracy: 70.53%
alpha=0.3 Average Class -1 Accuracy: 70.83%

alpha=0.4 Average Overall Accuracy: 69.88%
alpha=0.4 Average F1 Score (weighted avg): 72.28%
alpha=0.4 Average Class 1 Accuracy: 70.52%
alpha=0.4 Average Class -1 Accuracy: 67.50%

alpha=0.5 Average Overall Accuracy: 69.63%
alpha=0.5 Average F1 Score (weighted avg): 71.95%
a

In [19]:
"""fGW-kNN: alpha hyperparameter tuning"""

'fGW-kNN: alpha hyperparameter tuning'

In [10]:
k_list = [1,2,3,4,5,6,7,8,9,10]
num_folds = 10

In [11]:
alpha_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
alpha_tests_k = {
        alph: {}
        for alph in alpha_list
    }  # dictionary to store best k information for each alpha

for alpha in alpha_list:
    print()
    print("Currently testing alpha =", alpha)
    print()
    class_acc_dict, by_class_acc_dict, f1_score_dict, best_k_info = fgw_knn_alpha(distance_matrices, graphs_list, labels_list, num_folds, k_list=k_list, method="stratk",alpha=alpha)
    
    # Store the best k information for this alpha
    alpha_tests_k[alpha] = best_k_info


Currently testing alpha = 0.0

Stratified k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 66.66666666666666%, Fold 0
k=1, Class -1 Accuracy: 93.75%, Fold 0
k=1, Overall Classification Accuracy: 87.8048780487805%, Fold 0
k=1, F1 Score (weighted avg): 87.54%, Fold 0
k=2, Class 1 Accuracy: 66.66666666666666%, Fold 0
k=2, Class -1 Accuracy: 93.75%, Fold 0
k=2, Overall Classification Accuracy: 87.8048780487805%, Fold 0
k=2, F1 Score (weighted avg): 87.54%, Fold 0
k=3, Class 1 Accuracy: 55.55555555555556%, Fold 0
k=3, Class -1 Accuracy: 93.75%, Fold 0
k=3, Overall Classification Accuracy: 85.36585365853658%, Fold 0
k=3, F1 Score (weighted avg): 84.67%, Fold 0
k=4, Class 1 Accuracy: 55.55555555555556%, Fold 0
k=4, Class -1 Accuracy: 93.75%, Fold 0
k=4, Overall Classification Accuracy: 85.36585365853658%, Fold 0
k=4, F1 Score (weighted avg): 84.67%, Fold 0
k=5, Class 1 Accuracy: 55.55555555555556%, Fold 0
k=5, Class -1 Accuracy: 93.75%, Fold 0
k=5, Overall Classification Accuracy: 85.365853

In [12]:
# Find best alpha based on the best k performance for each alpha
best_alpha = alpha_list[0]
best_alpha_acc = 0
best_alpha_f1 = 0

print("\n" + "="*60)
print("SUMMARY OF BEST K PERFORMANCE FOR EACH ALPHA:")
print("="*60)

for alph in alpha_list:
    best_k_info = alpha_tests_k[alph]
    
    print(f"\nalpha={alph}:")
    print(f"  Best k: {best_k_info['best_k']}")
    print(f"  Overall Accuracy: {best_k_info['best_overall_accuracy']:.2f}%")
    print(f"  F1 Score (weighted avg): {best_k_info['best_f1_score']:.2f}%")
    print(f"  Class 1 Accuracy: {best_k_info['best_class_1_accuracy']:.2f}%")
    print(f"  Class -1 Accuracy: {best_k_info['best_class_minus1_accuracy']:.2f}%")

    # Check if this alpha gives the best overall accuracy
    if best_k_info['best_overall_accuracy'] > best_alpha_acc:
        best_alpha_acc = best_k_info['best_overall_accuracy']
        best_alpha = alph

    # Track best F1 score
    if best_k_info['best_f1_score'] > best_alpha_f1:
        best_alpha_f1 = best_k_info['best_f1_score']

print("\n" + "="*60)
print("OVERALL BEST PERFORMANCE:")
print("="*60)
best_info = alpha_tests_k[best_alpha]
print(f"Best alpha: {best_alpha}")
print(f"Best k for this alpha: {best_info['best_k']}")
print(f"Overall Accuracy: {best_info['best_overall_accuracy']:.2f}%")
print(f"F1 Score (weighted avg): {best_info['best_f1_score']:.2f}%")
print(f"Class 1 Accuracy: {best_info['best_class_1_accuracy']:.2f}%")
print(f"Class -1 Accuracy: {best_info['best_class_minus1_accuracy']:.2f}%")


SUMMARY OF BEST K PERFORMANCE FOR EACH ALPHA:

alpha=0.0:
  Best k: 6
  Overall Accuracy: 85.44%
  F1 Score (weighted avg): 84.20%
  Class 1 Accuracy: 50.14%
  Class -1 Accuracy: 95.00%

alpha=0.1:
  Best k: 6
  Overall Accuracy: 85.93%
  F1 Score (weighted avg): 84.67%
  Class 1 Accuracy: 50.14%
  Class -1 Accuracy: 95.62%

alpha=0.2:
  Best k: 7
  Overall Accuracy: 86.18%
  F1 Score (weighted avg): 84.64%
  Class 1 Accuracy: 46.39%
  Class -1 Accuracy: 96.88%

alpha=0.3:
  Best k: 6
  Overall Accuracy: 85.92%
  F1 Score (weighted avg): 84.41%
  Class 1 Accuracy: 47.64%
  Class -1 Accuracy: 96.25%

alpha=0.4:
  Best k: 6
  Overall Accuracy: 85.93%
  F1 Score (weighted avg): 84.43%
  Class 1 Accuracy: 47.50%
  Class -1 Accuracy: 96.25%

alpha=0.5:
  Best k: 6
  Overall Accuracy: 86.17%
  F1 Score (weighted avg): 84.80%
  Class 1 Accuracy: 48.75%
  Class -1 Accuracy: 96.25%

alpha=0.6:
  Best k: 1
  Overall Accuracy: 85.96%
  F1 Score (weighted avg): 85.89%
  Class 1 Accuracy: 66.81%
 

In [13]:
### Standardized feature vector experiments ###
graphs_list_norm = []
labels_list_norm = []
for graph_id, graph in graphs_normalized.items():
    # Add the graph object to the list
    graphs_list_norm.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list_norm.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices_norm = [compute_bzr_distance_matrix(graph) for graph in graphs_list_norm]

In [14]:
alpha_tests_norm = {
        alph: []
        for alph in alpha_list
    }  # dictionary to store accuracies for each k
for alpha in alpha_list:
    print()
    print("Currently testing alpha =", alpha)
    print()
    bzr_exp = fgw_barycenter_alg_alpha(distance_matrices_norm, 
                                       graphs_list_norm, 
                                       labels_list_norm, 
                                       num_folds=10, 
                                       num_nodes_negone=36, 
                                       num_nodes_one=36,
                                       alpha=alpha,
                                       method="stratk")
    #classification_accuracy = sum(bzr_exp[0])/len(bzr_exp[0]) * 100
    alpha_tests_norm[alpha].append(bzr_exp)


Currently testing alpha = 0.0

Stratified k-fold CV selected
Fold 0:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.27944779150212984  Fold 0
F1 Score (weighted avg): 66.51884700665188%, Fold 0
Classification Accuracy: 63.41463414634146% Fold 0
Classification Accuracy for Class -1: 62.5% Fold 0
Classification Accuracy for Class 1: 66.66666666666666% Fold 0
Fold 1:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.3045648365005551  Fold 1
F1 Score (weighted avg): 68.74296435272045%, Fold 1
Classification Accuracy: 65.85365853658537% Fold 1
Classification Accuracy for Class -1: 59.375% Fold 1
Classification Accuracy for Class 1: 88.88888888888889% Fold 1
Fold 2:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.32894575935993053  Fold 2
F1 Score (weighted avg): 67.81966001478197%, Fo

In [15]:
best_alpha = alpha_list[0]
best_alpha_acc = 0
best_alpha_f1 = 0
alph_avg_accuracies = {}
alph_avg_f1_scores = {}
classes = [-1, 1]

for alph in alpha_list:
        avg_acc = sum(alpha_tests_norm[alph][0][0]) / len(alpha_tests_norm[alph][0][0])
        avg_f1 = sum(alpha_tests_norm[alph][0][3]) / len(alpha_tests_norm[alph][0][3])

        alph_avg_accuracies[alph] = {
            'overall':
            avg_acc,
            'class_1':
            sum(alpha_tests_norm[alph][0][1]) / len(alpha_tests_norm[alph][0][1]),
            'class_-1':
            sum(alpha_tests_norm[alph][0][2]) / len(alpha_tests_norm[alph][0][2])
        }
        alph_avg_f1_scores[alph] = avg_f1

        print(f"\nalpha={alph} Average Overall Accuracy: {avg_acc:.2f}%")
        print(f"alpha={alph} Average F1 Score (weighted avg): {avg_f1:.2f}%")
        for class_label in [1, -1]:
            print(
                f"alpha={alph} Average Class {class_label} Accuracy: {alph_avg_accuracies[alph][f'class_{class_label}']:.2f}%"
            )

        if avg_acc > best_alpha_acc:
            best_alpha_acc = avg_acc
            best_alpha = alph

        if avg_f1 > best_alpha_f1:
            best_alpha_f1 = avg_f1

print(f"\nBest performance achieved with alpha={best_alpha}:")
print(f"Overall Accuracy: {alph_avg_accuracies[best_alpha]['overall']:.2f}%")
print(f"F1 Score (weighted avg): {alph_avg_f1_scores[best_alpha]:.2f}%")
print(f"Class 1 Accuracy: {alph_avg_accuracies[best_alpha]['class_1']:.2f}%")
print(f"Class -1 Accuracy: {alph_avg_accuracies[best_alpha]['class_-1']:.2f}%")


alpha=0.0 Average Overall Accuracy: 67.67%
alpha=0.0 Average F1 Score (weighted avg): 70.36%
alpha=0.0 Average Class 1 Accuracy: 67.41%
alpha=0.0 Average Class -1 Accuracy: 68.47%

alpha=0.1 Average Overall Accuracy: 69.15%
alpha=0.1 Average F1 Score (weighted avg): 71.63%
alpha=0.1 Average Class 1 Accuracy: 69.60%
alpha=0.1 Average Class -1 Accuracy: 67.50%

alpha=0.2 Average Overall Accuracy: 69.90%
alpha=0.2 Average F1 Score (weighted avg): 72.22%
alpha=0.2 Average Class 1 Accuracy: 71.16%
alpha=0.2 Average Class -1 Accuracy: 65.28%

alpha=0.3 Average Overall Accuracy: 69.64%
alpha=0.3 Average F1 Score (weighted avg): 71.93%
alpha=0.3 Average Class 1 Accuracy: 71.47%
alpha=0.3 Average Class -1 Accuracy: 62.92%

alpha=0.4 Average Overall Accuracy: 69.87%
alpha=0.4 Average F1 Score (weighted avg): 72.10%
alpha=0.4 Average Class 1 Accuracy: 71.47%
alpha=0.4 Average Class -1 Accuracy: 63.89%

alpha=0.5 Average Overall Accuracy: 69.40%
alpha=0.5 Average F1 Score (weighted avg): 71.57%
a

In [16]:
alpha_tests_k_norm = {
        alph: {}
        for alph in alpha_list
    }  # dictionary to store best k information for each alpha

for alpha in alpha_list:
    print()
    print("Currently testing alpha =", alpha)
    print()
    class_acc_dict, by_class_acc_dict, f1_score_dict, best_k_info = fgw_knn_alpha(distance_matrices_norm, 
                                                                                  graphs_list_norm, 
                                                                                  labels_list_norm, 
                                                                                  num_folds, 
                                                                                  k_list=k_list, 
                                                                                  method="stratk",alpha=alpha)
    
    # Store the best k information for this alpha
    alpha_tests_k_norm[alpha] = best_k_info


Currently testing alpha = 0.0

Stratified k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 66.66666666666666%, Fold 0
k=1, Class -1 Accuracy: 93.75%, Fold 0
k=1, Overall Classification Accuracy: 87.8048780487805%, Fold 0
k=1, F1 Score (weighted avg): 87.54%, Fold 0
k=2, Class 1 Accuracy: 66.66666666666666%, Fold 0
k=2, Class -1 Accuracy: 93.75%, Fold 0
k=2, Overall Classification Accuracy: 87.8048780487805%, Fold 0
k=2, F1 Score (weighted avg): 87.54%, Fold 0
k=3, Class 1 Accuracy: 55.55555555555556%, Fold 0
k=3, Class -1 Accuracy: 93.75%, Fold 0
k=3, Overall Classification Accuracy: 85.36585365853658%, Fold 0
k=3, F1 Score (weighted avg): 84.67%, Fold 0
k=4, Class 1 Accuracy: 55.55555555555556%, Fold 0
k=4, Class -1 Accuracy: 93.75%, Fold 0
k=4, Overall Classification Accuracy: 85.36585365853658%, Fold 0
k=4, F1 Score (weighted avg): 84.67%, Fold 0
k=5, Class 1 Accuracy: 55.55555555555556%, Fold 0
k=5, Class -1 Accuracy: 93.75%, Fold 0
k=5, Overall Classification Accuracy: 85.365853

In [17]:
# Find best alpha based on the best k performance for each alpha
best_alpha = alpha_list[0]
best_alpha_acc = 0
best_alpha_f1 = 0

print("\n" + "="*60)
print("SUMMARY OF BEST K PERFORMANCE FOR EACH ALPHA:")
print("="*60)

for alph in alpha_list:
    best_k_info = alpha_tests_k_norm[alph]
    
    print(f"\nalpha={alph}:")
    print(f"  Best k: {best_k_info['best_k']}")
    print(f"  Overall Accuracy: {best_k_info['best_overall_accuracy']:.2f}%")
    print(f"  F1 Score (weighted avg): {best_k_info['best_f1_score']:.2f}%")
    print(f"  Class 1 Accuracy: {best_k_info['best_class_1_accuracy']:.2f}%")
    print(f"  Class -1 Accuracy: {best_k_info['best_class_minus1_accuracy']:.2f}%")

    # Check if this alpha gives the best overall accuracy
    if best_k_info['best_overall_accuracy'] > best_alpha_acc:
        best_alpha_acc = best_k_info['best_overall_accuracy']
        best_alpha = alph

    # Track best F1 score
    if best_k_info['best_f1_score'] > best_alpha_f1:
        best_alpha_f1 = best_k_info['best_f1_score']

print("\n" + "="*60)
print("OVERALL BEST PERFORMANCE:")
print("="*60)
best_info = alpha_tests_k_norm[best_alpha]
print(f"Best alpha: {best_alpha}")
print(f"Best k for this alpha: {best_info['best_k']}")
print(f"Overall Accuracy: {best_info['best_overall_accuracy']:.2f}%")
print(f"F1 Score (weighted avg): {best_info['best_f1_score']:.2f}%")
print(f"Class 1 Accuracy: {best_info['best_class_1_accuracy']:.2f}%")
print(f"Class -1 Accuracy: {best_info['best_class_minus1_accuracy']:.2f}%")


SUMMARY OF BEST K PERFORMANCE FOR EACH ALPHA:

alpha=0.0:
  Best k: 4
  Overall Accuracy: 85.20%
  F1 Score (weighted avg): 84.38%
  Class 1 Accuracy: 54.86%
  Class -1 Accuracy: 93.43%

alpha=0.1:
  Best k: 4
  Overall Accuracy: 85.21%
  F1 Score (weighted avg): 83.90%
  Class 1 Accuracy: 50.28%
  Class -1 Accuracy: 94.69%

alpha=0.2:
  Best k: 1
  Overall Accuracy: 85.46%
  F1 Score (weighted avg): 85.25%
  Class 1 Accuracy: 64.44%
  Class -1 Accuracy: 91.24%

alpha=0.3:
  Best k: 1
  Overall Accuracy: 85.71%
  F1 Score (weighted avg): 85.52%
  Class 1 Accuracy: 64.44%
  Class -1 Accuracy: 91.55%

alpha=0.4:
  Best k: 1
  Overall Accuracy: 85.47%
  F1 Score (weighted avg): 85.33%
  Class 1 Accuracy: 65.56%
  Class -1 Accuracy: 90.93%

alpha=0.5:
  Best k: 1
  Overall Accuracy: 85.47%
  F1 Score (weighted avg): 85.36%
  Class 1 Accuracy: 65.42%
  Class -1 Accuracy: 90.93%

alpha=0.6:
  Best k: 1
  Overall Accuracy: 85.23%
  F1 Score (weighted avg): 84.98%
  Class 1 Accuracy: 64.44%
 